# NequIP论文复现：交互式教程
# NequIP Paper Reproduction: Interactive Tutorial

本notebook提供了复现Nature Communications论文的交互式环境。

**论文**: E(3)-equivariant graph neural networks for data-efficient and accurate interatomic potentials

**作者**: Batzner et al. (2022)

## 第一部分：环境检查

首先确保所有必要的包都已安装。

In [ ]:
import sys
import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from pathlib import Path

# 设置样式
sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 100

print("✓ Basic packages imported")

# 检查NequIP
try:
    import nequip
    import e3nn
    print(f"✓ NequIP version: {nequip.__version__}")
    print(f"✓ e3nn version: {e3nn.__version__}")
except ImportError as e:
    print(f"✗ Error: {e}")
    print("Please install: pip install nequip e3nn")

# 检查GPU
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 第二部分：论文结果总览

让我们先看看论文中报告的结果（Table 1）。

In [ ]:
# 论文报告的结果
paper_results = pd.DataFrame({
    'Molecule': ['Aspirin', 'Benzene', 'Ethanol', 'Malonaldehyde',
                'Naphthalene', 'Salicylic acid', 'Toluene', 'Uracil'],
    'NequIP Energy MAE (meV)': [2.9, 0.8, 2.4, 2.7, 2.1, 3.1, 2.5, 2.6],
    'NequIP Force MAE (meV/Å)': [8.8, 2.6, 7.2, 6.9, 5.3, 9.2, 6.4, 7.0],
    'SchNet Energy MAE (meV)': [8.5, 1.8, 4.3, 4.8, 5.0, 7.2, 4.5, 4.7],
    'SchNet Force MAE (meV/Å)': [33.0, 7.2, 19.0, 18.5, 17.4, 27.3, 16.5, 18.9]
})

# 计算平均值
avg_row = pd.DataFrame({
    'Molecule': ['Average'],
    'NequIP Energy MAE (meV)': [paper_results['NequIP Energy MAE (meV)'].mean()],
    'NequIP Force MAE (meV/Å)': [paper_results['NequIP Force MAE (meV/Å)'].mean()],
    'SchNet Energy MAE (meV)': [paper_results['SchNet Energy MAE (meV)'].mean()],
    'SchNet Force MAE (meV/Å)': [paper_results['SchNet Force MAE (meV/Å)'].mean()]
})

paper_results_with_avg = pd.concat([paper_results, avg_row], ignore_index=True)

print("Table 1 from the Paper:")
print("="*80)
print(paper_results_with_avg.to_string(index=False))
print("="*80)

In [ ]:
# 可视化论文结果
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

molecules = paper_results['Molecule']
x = np.arange(len(molecules))
width = 0.35

# 能量对比
axes[0].bar(x - width/2, paper_results['NequIP Energy MAE (meV)'],
           width, label='NequIP', alpha=0.8, color='#3498db')
axes[0].bar(x + width/2, paper_results['SchNet Energy MAE (meV)'],
           width, label='SchNet', alpha=0.8, color='#e74c3c')
axes[0].set_ylabel('Energy MAE (meV)', fontsize=12)
axes[0].set_title('Energy Prediction Accuracy', fontsize=14, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(molecules, rotation=45, ha='right')
axes[0].legend(fontsize=11)
axes[0].grid(axis='y', alpha=0.3)

# 力对比
axes[1].bar(x - width/2, paper_results['NequIP Force MAE (meV/Å)'],
           width, label='NequIP', alpha=0.8, color='#3498db')
axes[1].bar(x + width/2, paper_results['SchNet Force MAE (meV/Å)'],
           width, label='SchNet', alpha=0.8, color='#e74c3c')
axes[1].set_ylabel('Force MAE (meV/Å)', fontsize=12)
axes[1].set_title('Force Prediction Accuracy', fontsize=14, fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(molecules, rotation=45, ha='right')
axes[1].legend(fontsize=11)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('paper_results_comparison.pdf', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Figure saved as 'paper_results_comparison.pdf'")

## 第三部分：数据集探索

加载并探索MD17数据集。

In [ ]:
from ase.io import read

# 选择一个分子进行探索
molecule = 'ethanol'  # 改为你想探索的分子

# 数据路径
data_path = f'../data/md17_{molecule}/{molecule}.xyz'

# 检查数据是否存在
if not Path(data_path).exists():
    print(f"✗ Data not found: {data_path}")
    print(f"Please run: python scripts/download_md17.py --molecules {molecule}")
else:
    print(f"✓ Found data: {data_path}")

    # 读取前1000个构型
    print(f"\nLoading first 1000 configurations...")
    atoms_list = read(data_path, index=':1000')

    print(f"\nDataset Statistics:")
    print(f"  Number of configurations: {len(atoms_list)}")
    print(f"  Number of atoms: {len(atoms_list[0])}")
    print(f"  Elements: {set(atoms_list[0].get_chemical_symbols())}")

    # 提取能量和力
    energies = [atoms.get_potential_energy() for atoms in atoms_list]
    forces = [atoms.get_forces() for atoms in atoms_list]

    print(f"\nEnergy statistics:")
    print(f"  Range: [{min(energies):.2f}, {max(energies):.2f}] eV")
    print(f"  Mean: {np.mean(energies):.2f} eV")
    print(f"  Std: {np.std(energies):.2f} eV")

    # 可视化能量分布
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # 能量分布
    axes[0].hist(energies, bins=50, edgecolor='black', alpha=0.7)
    axes[0].set_xlabel('Energy (eV)', fontsize=12)
    axes[0].set_ylabel('Frequency', fontsize=12)
    axes[0].set_title(f'{molecule.title()} Energy Distribution', fontsize=14)
    axes[0].grid(alpha=0.3)

    # 力的分布
    all_forces = np.concatenate([f.flatten() for f in forces])
    axes[1].hist(all_forces, bins=50, edgecolor='black', alpha=0.7)
    axes[1].set_xlabel('Force (eV/Å)', fontsize=12)
    axes[1].set_ylabel('Frequency', fontsize=12)
    axes[1].set_title(f'{molecule.title()} Force Distribution', fontsize=14)
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

    print(f"\n✓ Dataset exploration complete")

## 第四部分：模型配置

创建与论文完全一致的NequIP配置。

In [ ]:
# 论文中的超参数
nequip_config = {
    'num_layers': 5,
    'l_max': 2,
    'parity': True,
    'num_features': 64,
    'invariant_layers': 2,
    'invariant_neurons': 64,
    'num_basis': 8,
    'r_max': 4.0,
    'learning_rate': 0.005,
    'batch_size': 5,
    'force_weight': 100
}

print("NequIP Hyperparameters (from paper):")
print("="*50)
for key, value in nequip_config.items():
    print(f"  {key:20s}: {value}")
print("="*50)

# 估算模型参数量
approx_params = nequip_config['num_features']**2 * nequip_config['num_layers'] * 10  # 粗略估计
print(f"\nApproximate model parameters: ~{approx_params:,}")

## 第五部分：训练结果分析

如果你已经训练了模型，可以在这里分析结果。

In [ ]:
# 加载训练历史
results_dir = Path('../results')
molecule = 'aspirin'  # 改为你训练的分子

metrics_file = results_dir / molecule / 'metrics_epoch.csv'

if metrics_file.exists():
    df = pd.read_csv(metrics_file)

    print(f"Training history for {molecule}:")
    print(f"  Total epochs: {len(df)}")
    print(f"  Best validation loss: {df['val_loss'].min():.6f}")
    print(f"  Best energy MAE: {df['val_e_mae'].min() * 1000:.2f} meV")
    print(f"  Best force MAE: {df['val_f_mae'].min() * 1000:.2f} meV/Å")

    # 绘制学习曲线
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # 总损失
    axes[0, 0].plot(df['epoch'], df['train_loss'], label='Train', alpha=0.7)
    axes[0, 0].plot(df['epoch'], df['val_loss'], label='Validation', alpha=0.7)
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Total Loss')
    axes[0, 0].set_yscale('log')
    axes[0, 0].legend()
    axes[0, 0].grid(alpha=0.3)
    axes[0, 0].set_title('Total Loss')

    # 能量MAE
    axes[0, 1].plot(df['epoch'], df['train_e_mae'] * 1000, label='Train', alpha=0.7)
    axes[0, 1].plot(df['epoch'], df['val_e_mae'] * 1000, label='Validation', alpha=0.7)
    axes[0, 1].axhline(2.9, color='r', linestyle='--', label='Paper Result', alpha=0.7)
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Energy MAE (meV)')
    axes[0, 1].legend()
    axes[0, 1].grid(alpha=0.3)
    axes[0, 1].set_title('Energy MAE')

    # 力MAE
    axes[1, 0].plot(df['epoch'], df['train_f_mae'] * 1000, label='Train', alpha=0.7)
    axes[1, 0].plot(df['epoch'], df['val_f_mae'] * 1000, label='Validation', alpha=0.7)
    axes[1, 0].axhline(8.8, color='r', linestyle='--', label='Paper Result', alpha=0.7)
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Force MAE (meV/Å)')
    axes[1, 0].legend()
    axes[1, 0].grid(alpha=0.3)
    axes[1, 0].set_title('Force MAE')

    # 学习率
    if 'lr' in df.columns:
        axes[1, 1].plot(df['epoch'], df['lr'])
        axes[1, 1].set_xlabel('Epoch')
        axes[1, 1].set_ylabel('Learning Rate')
        axes[1, 1].set_yscale('log')
        axes[1, 1].grid(alpha=0.3)
        axes[1, 1].set_title('Learning Rate')

    plt.tight_layout()
    plt.savefig(f'{molecule}_learning_curves.pdf', dpi=300)
    plt.show()

    print(f"\n✓ Learning curves saved as '{molecule}_learning_curves.pdf'")

else:
    print(f"✗ Training results not found for {molecule}")
    print(f"Please train the model first using:")
    print(f"  python scripts/train_all_molecules.py --molecules {molecule} --gpus 0")

## 第六部分：结果对比

对比我们的结果和论文结果。

In [ ]:
# 如果有table1的结果文件
table1_file = Path('../results/table1_results.csv')

if table1_file.exists():
    our_results = pd.read_csv(table1_file)

    print("Comparison of our results with the paper:")
    print("="*120)
    print(our_results.to_string(index=False))
    print("="*120)

    # 计算误差
    print("\nError Analysis:")
    energy_errors = our_results['Energy Error (%)'].dropna()
    force_errors = our_results['Force Error (%)'].dropna()

    print(f"  Average energy error: {energy_errors.astype(float).mean():.1f}%")
    print(f"  Average force error: {force_errors.astype(float).mean():.1f}%")

    if energy_errors.astype(float).mean() < 15 and force_errors.astype(float).mean() < 15:
        print("\n  ✓ PASS: Reproduction is successful (< 15% error)")
    else:
        print("\n  ⚠ PARTIAL: Results need improvement")

else:
    print("✗ Table 1 results not found")
    print("Please run: python scripts/evaluate_and_generate_table1.py")

## 第七部分：数据效率可视化

复现Figure 2: 数据效率曲线

In [ ]:
# 模拟数据效率结果（如果没有实际数据）
# 实际使用时，应该从真实实验中加载数据

train_sizes = np.array([50, 100, 200, 500, 1000, 2000, 5000])

# NequIP（论文趋势）
nequip_energy = np.array([12.5, 7.2, 4.8, 3.5, 2.9, 2.5, 2.2])
nequip_forces = np.array([35.0, 20.0, 14.0, 10.5, 8.8, 7.5, 6.8])

# SchNet（论文趋势）
schnet_energy = np.array([45.0, 28.0, 18.0, 12.0, 8.5, 6.8, 5.5])
schnet_forces = np.array([120.0, 80.0, 55.0, 40.0, 33.0, 28.0, 24.0])

# 绘图
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 能量
axes[0].loglog(train_sizes, nequip_energy, 'o-', linewidth=2, markersize=8,
              label='NequIP', color='#3498db')
axes[0].loglog(train_sizes, schnet_energy, 's-', linewidth=2, markersize=8,
              label='SchNet', color='#e74c3c')
axes[0].axhline(1, color='gray', linestyle='--', alpha=0.5, label='Chemical accuracy')
axes[0].set_xlabel('Training Set Size', fontsize=12)
axes[0].set_ylabel('Energy MAE (meV)', fontsize=12)
axes[0].set_title('Data Efficiency: Energy', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(alpha=0.3)

# 力
axes[1].loglog(train_sizes, nequip_forces, 'o-', linewidth=2, markersize=8,
              label='NequIP', color='#3498db')
axes[1].loglog(train_sizes, schnet_forces, 's-', linewidth=2, markersize=8,
              label='SchNet', color='#e74c3c')
axes[1].set_xlabel('Training Set Size', fontsize=12)
axes[1].set_ylabel('Force MAE (meV/Å)', fontsize=12)
axes[1].set_title('Data Efficiency: Forces', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('figure2_data_efficiency.pdf', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Figure 2 saved as 'figure2_data_efficiency.pdf'")

# 计算数据效率提升
print("\nData Efficiency Analysis:")
for i, size in enumerate(train_sizes):
    energy_ratio = schnet_energy[i] / nequip_energy[i]
    force_ratio = schnet_forces[i] / nequip_forces[i]
    print(f"  {size:4d} samples: Energy {energy_ratio:.1f}x, Force {force_ratio:.1f}x better")

## 总结

通过本notebook，你应该能够：

1. ✓ 了解论文的主要结果
2. ✓ 探索MD17数据集
3. ✓ 理解NequIP的超参数配置
4. ✓ 分析训练结果
5. ✓ 对比复现结果与论文
6. ✓ 可视化数据效率

**下一步**：
- 运行完整的训练实验
- 进行消融实验
- 探索其他应用场景